# Exploring Uncertainty in Climate Data

There are several kinds of scientific uncertainty that arise when working with long-term projections of future climates:
1. Model Uncertainty, which illustrates the differences between different models (namely, how model physics, settings, or parameters can change the outcome)
2. Scenario Uncertainty, which illustrates the differences in outcomes between end-of-century emissions targets
3. Internal Variability, which represents the variations inherent within the climate system itself

This notebook explores **Model Uncertainty** in the Cal-Adapt: Analytics Engine by focusing on **temperature trends** across simulations. We also compare the suite of models currently available in the [Cal-Adapt Data Catalog](https://analytics.cal-adapt.org/data/) to the full set of CMIP6 models to illustrate the differences between our models and all available models

---
**Story**

As a user, I want to understand when <span style="color:#FF0000">**taking a mean across models is not appropriate for the data**, <span style="color:#000000"> by visualizing the differences between models and see:
1. A range of possibilities for the end of century across all available CMIP6 models
2. Which models are provided in the Analytics Engine as compared to CMIP6 models
3. What the local response (postage stamps) is across models

**Step 0.1** -- Setup and CMIP6 processing (as needed)

**Step 0.2** -- Calculate key metrics needed: (1) Historical baseline (1850-1900); (2) Historical MMM; (3) SSP3-7.0 MMM

**Step 1** -- Illustrate range of temperature trends across CMIP6 archive, in *warming levels* context, with warming level designating line, all historical models + historical MMM, all future models + future MMM +  highlight the Analytics Engine models (one color per model?) amongst spread

**Step 2** -- Illustrate postage stamp view (spatial plots) of min/max/mean/median of CMIP6 for local response looks like
    
**Step 3** -- Illustrate example applications

---

## Step 0: Setup and CMIP6 data processing
Import useful packages and libraries. 

In [ ]:
# !pip install -r requirements.txt
import climakitae as ck
!pip install xmip
import xarray as xr
import pandas as pd
import numpy as np
import datetime
import intake
import dask
import matplotlib as mpl
import matplotlib.pyplot as plt
import xesmf as xe
import cartopy.crs as ccrs
import holoviews as hv
import hvplot.xarray # noqa
import hvplot.pandas

import warnings
warnings.filterwarnings("ignore")

Grab the regridded CMIP6 models that have both historical and SSP3-7.0 simulations

In [ ]:
from climakitae.selectors import Boundaries 
geographies = Boundaries()
us_states = geographies._us_states
us_counties = geographies._ca_counties 
us_watersheds = geographies._ca_watersheds

col = intake.open_esm_datastore("https://cadcat.s3.amazonaws.com/tmp/cmip6-regrid.json")

In [ ]:
from xmip.preprocessing import (
    rename_cmip6
) # will figure out how to avoid later

def cf_to_dt(ds):
    """convert cftime to pandas datetime"""
    if (
        type(ds.indexes['time']) not in 
        [pd.core.indexes.datetimes.DatetimeIndex]
    ): 
        datetimeindex = ds.indexes['time'].to_datetimeindex()
        ds['time'] = datetimeindex
    return ds

def calendar_align(ds):
    '''
    different models have different calendars
    (some no leap, some 360 day). this results
    in a huge dataset with lots of empty
    values in time when concatenated. 
    the following function sets the day for all monthly
    values to the 1st of each month -
    WARNING this can impact functions which use
    the number of days in each month (eg
    precip flux to total monthly accumulation).
    '''
    ds['time'] = pd.to_datetime(ds.time.dt.strftime('%Y-%m'))
    return ds

def wrapper(ds):
    ds_simulation = ds.attrs["source_id"]
    ds_scenario = ds.attrs["experiment_id"]
    ds_freq = ds.attrs["frequency"]    

    ds = rename_cmip6(ds) # will figure out alternative
    ds = cf_to_dt(ds)
    if ds_freq in ('mon'): 
        ds = calendar_align(ds)
    ds = ds.drop_vars(["lon","lat","height"],
                     errors="ignore")
    ds = ds.assign_coords({'simulation' : 
                           ds_simulation,
                          'scenario' :
                          ds_scenario})
    ds = ds.squeeze(drop=True)
    
    return ds

In [ ]:
class cmip_opt():
    def __init__(self, variable='tas', 
                  area_subset='states', 
                 location='California',
                 timescale='monthly',
                area_average=True):
        self.variable = variable
        self.area_subset = area_subset
        self.location = location
        self.area_average = area_average
        self.timescale = timescale
        
    def cmip_clip(self, ds):
        variable = self.variable
        location = self.location
        area_average = self.area_average
        area_subset = self.area_subset
        timescale = self.timescale
        
        to_drop = [v for v in list(
                    ds.data_vars) 
                  if v != variable]
        ds = ds.drop_vars(to_drop)
        ds = clip_region(ds,area_subset,location)
        if area_average:
            ds = area_wgt_average(ds)
        return ds
    
def clip_region(ds,area_subset,location):
    """
    clips CMIP6 dataset using a polygon.
    ds is the dataset
    state is a string ("California")
    (check catalog for other names)
    opt = 'True' to burn in all cells
    which touch the boundary (keep as False)
    """
    ds = ds.rio.write_crs(4326)
    
    if 'counties' in area_subset:
        ds_region = us_counties[us_counties.NAME 
                    == location].geometry
    elif 'states' in area_subset:
        ds_region = us_states[us_states.NAME 
                    == location].geometry
        
    try:
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=False)
    except: 
        # if no grid centers in region
        # instead select all cells which 
        # intersect the region
        print('selecting all cells which intersect region')
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=True)
        
    return ds

def area_wgt_average(ds):
    weights = np.cos(np.deg2rad(ds.y))
    weights.name = "weights"
    ds_weighted = ds.weighted(weights)
    ds = ds_weighted.mean(("x","y"))
    return ds

def drop_member_id(dset_dict): 
    """Drop member_id coordinate/dimensions 
    Args: 
        dset_dict (dict): dictionary in the format {dataset_name:xr.Dataset}
    """
    for dname, dset in dset_dict.items():
        if "member_id" in dset.coords: 
            dset = dset.isel(member_id=0).drop("member_id") # Drop coord
            dset_dict.update({dname: dset}) # Update dataset in dictionary
    return dset_dict

In [ ]:
# data selection options
copt = cmip_opt()
copt.variable = 'tas'
copt.area_subset = 'states' 
copt.location = 'California'
copt.area_average = False
copt.timescale = 'Amon'

In [ ]:
# grab data from the cmip6 archive
cat = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    member_id = "r1i1p1f1",
    require_all_on="source_id"
).search(activity_id = ["CMIP","ScenarioMIP"])

dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper)

# grab the additional cal-adapt simulations
paths = ['CESM2.*r11i1p1f1', 'CNRM-ESM2-1.*r1i1p1f2', 'MPI-ESM1-2-LR.*r7i1p1f1']
cat = col.search(table_id=copt.timescale, 
           variable_id=copt.variable, 
           path=paths, 
           activity_id=['CMIP', 'ScenarioMIP'])

cal_dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper)

In [ ]:
# subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}
cal_hist_dsets = {key: val for key,val in cal_dsets.items()
             if "historical" in key}

# subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}
cal_ssp_dsets = {key: val for key,val in cal_dsets.items()
               if "ssp370" in key}

# drop member id 
hist_dsets = drop_member_id(hist_dsets)
cal_hist_dsets = drop_member_id(cal_hist_dsets)
ssp_dsets = drop_member_id(ssp_dsets) 
cal_ssp_dsets = drop_member_id(cal_ssp_dsets)

# merge dataset together
all_hist_mdls = hist_dsets | cal_hist_dsets
all_ssp_mdls = ssp_dsets | cal_ssp_dsets

hist_ds = xr.concat(list(all_hist_mdls.values()), dim='simulation').squeeze()
hist_ds = copt.cmip_clip(hist_ds.sel(time=slice('1850','2014')))

ssp_ds = xr.concat(list(all_ssp_mdls.values()), dim='simulation').squeeze()
ssp_ds = copt.cmip_clip(ssp_ds)

# concatenate data based on simulation
mdls_ds = xr.concat(
    [hist_ds, ssp_ds],
    dim="time",
    coords="minimal",
    compat="override",
    join="inner",
)
mdls_ds

## Step 1: Assess CMIP6 multi-model spread

In [ ]:
def cmip_annual(ds):
    """Processes CMIP6 dataset into annual smoothed timeseries"""
    ds_degC = ds - 273.15 # convert to degC
    ds_degC = ds_degC.groupby("time.year").mean(dim=["x","y"])
    ds_degC = ds_degC.groupby("time.year").mean(dim="time")
    return ds_degC

cmip_ds_yr = cmip_annual(mdls_ds).compute()
cmip_ds_yr.tas.attrs['units'] = '°C' # set units to Celsius

In [ ]:
# warming levels context [see explore_warming.ipynb]
def calc_anom(ds):
    """
    Calculates the temperature change relative to a historical baseline (1850-1900) for each model.
    Returns the difference from the input ds and the respective model baseline.
    """
    mdl_baseline = cmip_ds_yr.sel(year=slice(1850,1900)).mean("year")
    mdl_temp_anom = ds - mdl_baseline
    return mdl_temp_anom
    
def cmip_mmm(ds):
    """Calculate CMIP6 multi-model mean."""
    ds_mmm = ds.mean("simulation")
    return ds_mmm

cmip_anom = calc_anom(cmip_ds_yr)
cmip_anom_mmm = cmip_mmm(cmip_anom)

# cal-adapt
hist_start, hist_end = 1950, 2014
ssp_end = 2100
cae_mdls_ls = ["FGOALS-g3", "EC-Earth3-Veg", "CESM2", "CNRM-ESM2-1", "MPI-ESM1-2-LR"]
cae_mdls = cmip_anom.sel(simulation=cae_mdls_ls)
cae_anom = cae_mdls.sel(year=slice(hist_start, ssp_end))

# historical
hist_anom = cmip_anom.sel(year=slice(hist_start, hist_end))
hist_anom_mmm = cmip_anom_mmm.sel(year=slice(hist_start, hist_end))

# future
ssp_anom = cmip_anom.sel(year=slice(hist_end, ssp_end))
ssp_anom_mmm = cmip_anom_mmm.sel(year=slice(hist_end, ssp_end))

Next, plot the temperature spread amongst the members of the CMIP6 archive. 

We also highlight the Cal-Adapt: Analytics Engine models in order to illustrate where these models fall within the larger CMIP6 model spread.

In [ ]:
# figure set-up
h_color = 'grey'
ssp_color = 'orange'
cae_color = 'blue'
lw = 0.75
alpha = 0.25
ylab = hist_anom.tas.long_name + ' (' + hist_anom.tas.attrs['units'] + ')'

# all individual models
all_hist = hist_anom.hvplot.line(x="year", ylabel=ylab, xlabel='', by='simulation', line_width=lw, color=h_color, legend=False, alpha=alpha)
all_ssp = ssp_anom.hvplot.line(x="year", by="simulation", line_width=lw, color=ssp_color, legend=False, alpha=alpha)

# cal-adapt models
all_cae = cae_anom.hvplot.line(x="year", by="simulation", line_width=lw, color=cae_color, alpha=alpha*1.5)

# multi-model means
mmm_hist = hist_anom_mmm.hvplot.line(x="year", line_width=lw*3, color='black')
mmm_ssp = ssp_anom_mmm.hvplot.line(x="year", line_width=lw*3, color='red',
                                                       title="CMIP6 California mean surface temperature change relative to 1850-1900")
# warming level line
warm_level = 3.0
warmlevel_line = hv.HLine(warm_level).opts(color="black", line_width=1.0) \
                * hv.Text(x=1968, y=warm_level+0.45, text=str(warm_level) + "°C warming level").opts(style=dict(text_font_size='8pt'))

warmlevel_line * all_hist * all_ssp * all_cae * mmm_hist * mmm_ssp 

Many things are happening in this figure. Let's break it down:
- The thin grey lines represent a single CMIP6 model in the historical (1950-2014) period. Their corresponding future (SSP 3-7.0) counterparts are illustrated by the thin orange lines for 2014-2100. 
- The thick black line represents the multi-model mean for the **historical** period
- The thick red line represents the multi-model mean for the **future** period
- The five thin blue lines represent the models currently downscaled models available in the *Cal-Adapt: Analytics Engine*
- The selected warming level is also provided. Play around with different values to see how it changes the response in the cells below

## Step 2: Illustrate spatial statistics

Next, let's spatially visualize the differences between the CMIP6 model archive at a designated warming level. We will also identify how the Cal-Aadapt: Analytics Engine models compares to the broader spread. 

First, identify where each individual model reaches the designated warming level. We will look specificallly at a 3°C warming level. 

In [ ]:
# calculate spatial data first
hist_xy_year = hist_ds.groupby("time.year").mean(dim="time")
hist_xy_year = hist_xy_year - 273.15
hist_xy_anom = calc_anom(hist_xy_year)

ssp_xy_year = ssp_ds.groupby("time.year").mean(dim="time")
ssp_xy_year = ssp_xy_year - 273.15
ssp_xy_anom = calc_anom(ssp_xy_year)

In [ ]:
# find the year each warming level is reached
warm_level = 3.0

# determine where each model reaches the future warming level
da_list = []
year_reached_by_sim = []
for simulation in ssp_anom.simulation.values: 
    data_to_use = ssp_anom.tas.sel(simulation=simulation)
    years_above_warmlevel = data_to_use.where(data_to_use > warm_level, drop=True)
    if len(years_above_warmlevel.year.values) < 1: 
        print("Warming level not reached for {0}".format(simulation))
    else: 
        year_warmlevel_reached = years_above_warmlevel.year.values[0]
        year_reached_by_sim.append((simulation, year_warmlevel_reached))
        da_list.append(ssp_xy_anom.sel(year=year_warmlevel_reached, simulation=simulation))
    
thresh_df = pd.DataFrame(
    data=year_reached_by_sim, 
    columns=["simulation","year_warming_level_reached"]
)
data_by_warmlevel = xr.concat(da_list, dim="simulation")

In [ ]:
# set up for plots
from climakitae.utils import _read_ae_colormap
cmap = _read_ae_colormap(cmap='ae_diverging', cmap_hex=True) # sets to a diverging colormap

def _make_hvplot(data, title, clim, sopt, width=200, height=200):
    """Make single map"""
    _plot = data.hvplot.image(
        x="x",
        y="y",
        grid=True,
        width=width,
        height=height,
        xaxis=None,
        yaxis=None,
        symmetric=sopt,
        clim=(vmin, vmax),
        title=title,
        clabel="Air Temperature (°C)",
        features=["coastline"],
        cmap=cmap,
    )
    return _plot

def _compute_vmin_vmax(da_min, da_max):
    """Compute min, max, and center for plotting"""
    vmin = np.nanpercentile(da_min, 1)
    vmax = np.nanpercentile(da_max, 99)
    # define center for diverging symmetric data
    if (vmin < 0) and (vmax > 0):
        # dabs = abs(vmax) - abs(vmin)
        sopt = True
    else:
        sopt = None
    return vmin, vmax, sopt

num_simulations = len(data_by_warmlevel.simulation.values) # number of simulations

# compute 1% min and 99% max of all simulations
vmin_l, vmax_l = [], []
for sim in range(num_simulations):
    data = data_by_warmlevel.isel(simulation=sim)
    vmin_i, vmax_i, sopt = _compute_vmin_vmax(data.tas, data.tas)
    vmin_l.append(vmin_i)
    vmax_l.append(vmax_i)
vmin = min(vmin_l)
vmax = max(vmax_l)

Now, let's plot the difference (anomaly) between the year that the warming level is reached and the 1850-1900 historical baseline for each simulation. 

Please note, this will take a few minutes to display.

In [ ]:
all_plots = _make_hvplot(  # plot first simulation separate from the loop
        data=data_by_warmlevel.isel(simulation=0),
        title=data_by_warmlevel.isel(simulation=0).simulation.item(),
        clim=(vmin, vmax),
        sopt=sopt)

for sim_i in range(1, num_simulations): # plot remaining simulations
    pl_i = _make_hvplot(
        data=data_by_warmlevel.isel(simulation=sim_i),
        title=data_by_warmlevel.isel(simulation=sim_i).simulation.item(),
        clim=(vmin, vmax),
        sopt=sopt)
    all_plots += pl_i

# additional aesthetic settings to tidy figure
all_plots.cols(5)  # organize columns
all_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
all_plots.opts(toolbar="below")  # set toolbar location
all_plots.opts(
            title="Air Temperature at 2m: Anomalies for "
            + str(warm_level)
            + "°C Warming by Simulation"
        )  # add title

all_plots

Next, let's see visualize the minimum/maximum/median/mean conditions across models. These statistics are calculated from the data observed in the figure above at each grid cell. 

In [ ]:
# compute stats
min_data = data_by_warmlevel.min(dim="simulation")
max_data = data_by_warmlevel.max(dim="simulation")
med_data = data_by_warmlevel.median(dim="simulation")
mean_data = data_by_warmlevel.mean(dim="simulation")

# set up plots
min_plot = _make_hvplot(
    data=min_data,
    title="Minimum",
    clim=(vmin, vmax),
    sopt=sopt)

max_plot = _make_hvplot(
    data=max_data,
    title="Maximum",
    clim=(vmin, vmax),
    sopt=sopt)

med_plot = _make_hvplot(
    data=med_data,
    title="Median",
    clim=(vmin, vmax),
    sopt=sopt)

mean_plot = _make_hvplot(
    data=mean_data,
    title="Mean",
    clim=(vmin, vmax),
    sopt=sopt)

In [ ]:
all_plots = mean_plot + med_plot + min_plot + max_plot

# additional aesthetic settings to tidy figure
all_plots.opts(toolbar="below")  # set toolbar location
all_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
all_plots.opts(
            title="Air Temperature at 2m: Anomalies for "
            + str(warm_level)
            + "°C Warming Across Models"
        )  # add title
    
all_plots

## Step 3: Example applications

What might be most useful to know when addressing model uncertainty in climate data, is to know when it is **not appropriate to use an average across models**, and when it is **reasonable to do so** (note this depends on the question being asked)

First, we will assess when it is not appropriate to use a multi-model mean in climate data. 

#### Example 1: "I want to know the exact temperature it will be at my location in 2100."

In [ ]:
# select my location
my_loc = ssp_xy_anom.sel(x=slice(-119, -118), y=slice(37.0, 38.0))
my_loc = my_loc.mean(dim=["x","y"])
my_loc.tas.attrs['units'] = '°C'

# calculate the multi-model mean for my location
my_loc_mmm = cmip_mmm(my_loc)

In [ ]:
# calculate standard deviations
mmm_1std_above = my_loc_mmm + my_loc_mmm.std(dim='year')
mmm_1std_below = my_loc_mmm - my_loc_mmm.std(dim='year')

# visualize the future model response and the multi-model mean
zero_line = hv.HLine(0.0).opts(color="black", line_width=1.0, line_dash="dashed")
loc_mdls = my_loc.hvplot.line(x="year", by="simulation", line_width=lw, color=ssp_color, alpha=alpha, legend=False,
                  title="Future mean surface temperature response at My Location")

loc_mmm = my_loc_mmm.hvplot.line(x="year", line_width=lw*4, color='r', legend=False)
std_above = mmm_1std_above.hvplot.line(x="year", line_width=lw*2, color='k', legend=False, alpha=alpha*1.5)
std_below = mmm_1std_below.hvplot.line(x="year", line_width=lw*2, color='k', legend=False, alpha=alpha*1.5)

loc_mdls * loc_mmm * zero_line * std_above * std_below

There are additional new elements being displayed in this figure: 
- The dashed line provides a reference for 0°C, primarily to help identify which models indicate a lower/cooler response in 2100, and which are higher/warmer. 
- The two solid lines around the multi-model mean (thick red line) represent one standard deviation around the multi-model mean. By providing the standard deviation in addition to the multi-model mean, we can further highlight which models indicate values outside of this statistical likelihood range. 

Notice, that at 2100, **some models have negative values**, while **others are positive** (around the dashed line).

Next, let's determine the range of values in 2100. 

In [ ]:
print('Highest model temperature at 2100: ', np.nanmax(my_loc.tas.sel(year=2100).values), '°C')
print('Multi-model mean temperature at 2100: ', my_loc_mmm.tas.sel(year=2100).values, '°C')
print('Lowest model temperature at 2100: ', np.nanmin(my_loc.tas.sel(year=2100).values), '°C')

With **more than a 7°C** difference between models, and many models falling outside the standard deviation range, it is therefore **not appropriate** to utilize the multi-model mean in this example. Each of these simulations represent a realistic climate future, and there is too much variation to confidently say the multi-model mean represents the overall climate conditions. It is **strongly recommended** that the full range of climate possibilities be considered; a single value cannot represent this range accurately. 

Now let's look at an example of when it is scientifically reasonable to take an average across models in climate data. 

#### Example 2: "Will Southern California be warmer in the future than it was in the past?"

In [ ]:
# look at southern california
lower_lat = 33.0
upper_lat = 37.0

def socal_area(ds):
    area = ds.sel(y=slice(lower_lat, upper_lat))
    return area

socal_hist = socal_area(hist_xy_anom).sel(year=slice(1981,2010))
socal_ssp = socal_area(ssp_xy_anom).sel(year=slice(2071,2100))

In [ ]:
# visualize the historical and future periods
socal_hist_area = socal_hist.mean(dim=['year','simulation'])
socal_ssp_area = socal_ssp.mean(dim=['year','simulation'])

hist_plot = _make_hvplot(
    data=socal_hist_area,
    title="Historical Mean",
    clim=(vmin, vmax),
    sopt=sopt)

future_plot = _make_hvplot(
    data=socal_ssp_area,
    title="Future Mean",
    clim=(vmin, vmax),
    sopt=sopt)

all_plots = hist_plot + future_plot

# additional aesthetic settings to tidy figure
all_plots.opts(toolbar="below")  # set toolbar location
all_plots.opts(hv.opts.Layout(merge_tools=True))  # merge toolbar
all_plots.opts(
            title="Air Temperature at 2m: Anomalies for "
            + str(warm_level)
            + "°C Warming Across Models"
        )  # add title
    
all_plots

In [ ]:
# calculate the area average and the multi-model mean for S.  California
socal_hist_yr = socal_hist.mean(dim=["x","y"])
socal_ssp_yr = socal_ssp.mean(dim=["x","y"])
socal_hist_yr.tas.attrs['units'] = '°C'
socal_ssp_yr.tas.attrs['units'] = '°C'

socal_hist_mmm = cmip_mmm(socal_hist_yr)
socal_ssp_mmm = cmip_mmm(socal_ssp_yr)

In [ ]:
# visualize historical model response and the multi-model mean
socal_hist_mdls = socal_hist_yr.hvplot.line(x='year', by='simulation', line_width=lw, color=h_color, alpha=alpha, legend=False)
hist_mmm = socal_hist_mmm.hvplot.line(x='year', line_width=lw*3, color='black', legend=False,
                                     title='Historical S. California mean surface temperature change relative to 1850-1900')

socal_hist_mdls * hist_mmm

In [ ]:
# visualize future model response and the multi-model mean
socal_ssp_mdls = socal_ssp_yr.hvplot.line(x='year', by='simulation', line_width=lw, color=ssp_color, alpha=alpha, legend=False)
ssp_mmm = socal_ssp_mmm.hvplot.line(x='year', line_width=lw*3, color='r', legend=False,
                                   title='Future S. California mean surface temperature change relative to 1850-1900')

socal_ssp_mdls * ssp_mmm

Lastly, let's establish the diffence between the multi-model mean in the historical (1981-2010) and future (2071-2100) periods. 

In [ ]:
hist_mmm = np.asarray(socal_hist_mmm.tas.values)
ssp_mmm = np.asarray(socal_ssp_mmm.tas.values)

print('Smallest difference between the future and historical multi-model means: ', np.min(ssp_mmm - hist_mmm), '°C')
print('Largest difference between the future and historical multi-model means: ', np.max(ssp_mmm - hist_mmm), '°C')

Using the individual models, the multi-model mean, and this min-max range, we can see that the future period consistently is higher than in the historical period for S. California. Thus, we can definitively say **yes, there will be a higher temperature in the future** in this example. 

However, if we wanted to know <span style="color:#FF0000">*how much warmer in the future*<span style="color:#000000">, we **must** consider the full range of climate possibilities and cannot use only the multi-model mean. There are too many possibilities! 

Want to know more about different kinds of climate uncertainty in climate data? Check out the `explore_precipitatation_uncertainty.ipynb` notebook too!